# Full Legacy Prediction Pipeline
This notebook demonstrates end-to-end prediction, diagnostics, and output visualization.

## 1. Load Data

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from IPython.display import display

# Use your private_cases.csv or .json
private_path = 'private_cases.json'
import json
with open(private_path) as f:
    data = json.load(f)
if isinstance(data, list):
    df = pd.DataFrame([r['input'] if 'input' in r else r for r in data])
else:
    df = pd.DataFrame([data['input']])
display(df.head())

## 2. Define Hybrid Logic

In [ ]:
def predict_5day(days, miles, receipts):
    base = 100 * days
    mileage = 0.5 * miles if miles < 100 else 0.3 * miles + 20
    bonus = 100
    capped_receipts = min(receipts, 400)
    return round(base + mileage + capped_receipts + bonus, 2)

def predict_short_trip(days, miles, receipts):
    return round(120 * days + 0.7 * miles + min(receipts, 300), 2)

def predict_high_receipt(days, miles, receipts):
    return round(110 * days + 0.4 * miles + min(receipts, 350) - 50, 2)

def predict_baseline(days, miles, receipts):
    return round(100 * days + 0.5 * miles + min(receipts, 300), 2)

def predict_dispatch(row):
    days = int(row['trip_duration_days'])
    miles = float(row['miles_traveled'])
    receipts = float(row['total_receipts_amount'])
    if days == 5:
        return predict_5day(days, miles, receipts), '5-day rule'
    elif miles < 100:
        return predict_short_trip(days, miles, receipts), 'short trip'
    elif receipts > 500:
        return predict_high_receipt(days, miles, receipts), 'high receipts'
    else:
        return predict_baseline(days, miles, receipts), 'baseline'

## 3. Run Prediction, Capture Intermediate Results

In [ ]:
preds = df.apply(lambda row: predict_dispatch(row), axis=1)
df['prediction'] = [p[0] for p in preds]
df['logic_used'] = [p[1] for p in preds]
display(df.head())

## 4. Visualize Logic Application

In [ ]:
logic_counts = df['logic_used'].value_counts()
logic_counts.plot(kind='bar', title='Logic Used Distribution', color='skyblue')
plt.ylabel('Count')
plt.show()

## 5. Visualize and Analyze Predictions

In [ ]:
# Distribution
plt.hist(df['prediction'], bins=20, color='orange')
plt.title('Prediction Output Distribution')
plt.xlabel('Predicted Output')
plt.ylabel('Count')
plt.show()

# Boxplot
df.boxplot(column='prediction', by='logic_used', grid=False, figsize=(8,4))
plt.title('Prediction by Logic Used')
plt.suptitle('')
plt.xlabel('Logic Used')
plt.ylabel('Prediction')
plt.show()

# Summary Stats
display(df.groupby('logic_used')['prediction'].describe())